# Your Pretext Task Has a Shortcut

**Topic:** Self-supervised shortcuts &nbsp;·&nbsp; **Difficulty:** Medium

Rotation prediction is a classic pretext task: rotate an image by 0, 90, 180, or
270 degrees and train a network to say which. It only teaches useful features if
the *only* way to answer is to understand the content.

These images have a one-pixel border artifact, a brighter left edge from the
capture pipeline. Rotating moves the bright edge somewhere else, so a model can
read the answer off four border means and learn nothing. Detect the shortcut,
then measure how strong it is.

## The data (created in the next cell)

- `base` `(1200, 24, 24)`: greyscale, with a brighter first column.
- Rotations applied with `np.rot90`. Labels are the rotation index.
- `X_rot`, `y_rot`: the rotated images and labels. `Xte`, `yte`: the held-out test split.

## Your task

With **sklearn and NumPy**:

1. `acc_border`: fit `LogisticRegression(max_iter=1000)` using **only the 4
   corner-adjacent border means** (mean of the first row, last row, first column,
   last column) as features, and score on the test split. If this is high, the
   shortcut exists.
2. `acc_center`: the same classifier using only the central `12x12` crop
   (flattened), which contains no border.
3. `acc_full`: using the whole flattened image.
4. Set `answers` to a dict with `"acc_border"`, `"acc_center"`, `"acc_full"`,
   and `"shortcut_strength"` = `acc_border - 0.25` (accuracy above chance from
   the border alone).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The exact-value check
assumes you fit `LogisticRegression(max_iter=1000)` on the raw features exactly as
written, with no scaling or reweighting. The evaluation cell checks the border
alone is far above chance while the center crop is near it.

In [ ]:
import numpy as np

rng = np.random.default_rng(3105)
n = 1200
base = rng.normal(0.5, 0.15, (n, 24, 24))
for i in range(n):                       # smooth, contentless texture
    base[i] = (base[i] + np.roll(base[i], 1, 0) + np.roll(base[i], 1, 1)) / 3
base[:, :, 0] += 0.45                    # the capture artifact

labels = rng.integers(0, 4, n)
X_rot = np.stack([np.rot90(base[i], labels[i]) for i in range(n)])
y_rot = labels

split = 900
Xtr, ytr = X_rot[:split], y_rot[:split]
Xte, yte = X_rot[split:], y_rot[split:]
print(f"{n} images, 4 rotations, {split} train / {n - split} test")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("acc_border", [], 1.0, -0.16743801481449916, 1e-06),
    ("acc_center", [], 0.31333333333333335, -0.052463911308543075, 1e-06),
    ("acc_full", [], 1.0, -0.16743801481449916, 1e-06),
    ("shortcut_strength", [], 0.75, -0.12557851111087437, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "acc_border" in answers:
    _short = answers["acc_border"] > 0.8
    _clean = answers["acc_center"] < 0.45
    print(f"  {'border alone solves it':<24} {'ok' if _short else 'wrong'}  "
          f"({answers['acc_border']:.3f} vs chance 0.25)")
    print(f"  {'center crop is chance':<24} {'ok' if _clean else 'wrong'}  "
          f"({answers['acc_center']:.3f})")
    _ok = _ok and _short and _clean

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

The border means are four numbers, and the rotation permutes them cyclically, so
a linear model on those four features recovers the label almost perfectly while
knowing nothing about the image. The center crop has the artifact cropped away
and scores near chance, which confirms the images contain no rotation-relevant
content at all. The lesson generalizes: before trusting a self-supervised
objective, try to solve it with a deliberately impoverished model, and if you
succeed the task is teaching a shortcut.

```python
import numpy as np
from sklearn.linear_model import LogisticRegression

def border_feats(A):
    return np.column_stack([A[:, 0, :].mean(1), A[:, -1, :].mean(1),
                            A[:, :, 0].mean(1), A[:, :, -1].mean(1)])

def center_feats(A):
    return A[:, 6:18, 6:18].reshape(len(A), -1)

def score(ftr, fte):
    m = LogisticRegression(max_iter=1000).fit(ftr, ytr)
    return float((m.predict(fte) == yte).mean())

acc_border = score(border_feats(Xtr), border_feats(Xte))
acc_center = score(center_feats(Xtr), center_feats(Xte))
acc_full = score(Xtr.reshape(len(Xtr), -1), Xte.reshape(len(Xte), -1))

answers = {
    "acc_border": acc_border,
    "acc_center": acc_center,
    "acc_full": acc_full,
    "shortcut_strength": float(acc_border - 0.25),
}
```